# Capítulo 9 · Ajuste fino con LoRA

Ajustar Qwen2.5-0.5B-Instruct con LoRA para que responda con el estilo de Ciclos Norte (saludo, respuesta breve y firma), y medir antes y después qué aprende: el estilo, muy bien; los datos, poco y mal.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. Con GPU, el ajuste tarda menos de un minuto. Sin GPU, cada paso de 8 ejemplos tarda unos 30 segundos: el cuaderno usa entonces 60 ejemplos y una época (unos cuatro minutos) y no carga el modelo grande de referencia.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Instalar PEFT y cargar

In [ ]:
!pip install -q peft

In [ ]:
"""Capítulo 9: ajuste fino con LoRA. Mejor con GPU; en CPU, con menos datos."""
import json
import random
import re
import time
import urllib.request
from pathlib import Path

import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_AJUSTE = "Qwen/Qwen2.5-0.5B-Instruct"   # el que ajustamos
MODELO_REFERENCIA = "Qwen/Qwen2.5-1.5B-Instruct"
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"
SEMILLA = 0
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"



def cargar(nombre):
    """Un modelo y su tokenizador, en la GPU si la hay."""
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    return tok, modelo.to(DISPOSITIVO)


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
CON_GPU = DISPOSITIVO == "cuda"
VARIANTES, EPOCAS = (3, 3) if CON_GPU else (1, 1)
print("GPU:", CON_GPU, "· variantes por pregunta:", VARIANTES, "· épocas:", EPOCAS)

## 2. Los datos de entrenamiento

In [ ]:
SALUDO, FIRMA = "¡Hola!", "Un saludo,\nEl equipo de Ciclos Norte"


def con_estilo(respuesta):
    """La respuesta con el estilo de la tienda: saludo, respuesta y firma."""
    return f"{SALUDO} {respuesta}\n\n{FIRMA}"


def ejemplos_de_entrenamiento(variantes=3):
    """Pares pregunta-respuesta a partir de las preguntas frecuentes: cada
    pregunta (la original y la consulta del capítulo 4) se escribe de varias
    formas, y su respuesta lleva el estilo de la tienda."""
    faq = descargar("preguntas_frecuentes.json")
    respuestas = {p["id"]: p["respuesta"] for p in faq["preguntas"]}
    preguntas = ([(p["pregunta"], p["id"]) for p in faq["preguntas"]]
                 + [(c, i) for c, i in faq["consultas"]])
    antes = ["", "Hola, ", "Buenas tardes. ", "Una consulta: ", "Buenos días, "]
    despues = ["", " Gracias.", " Un saludo.", " Muchas gracias."]
    azar = random.Random(SEMILLA)
    pares = []
    for pregunta, ident in preguntas:
        for _ in range(variantes):
            texto = azar.choice(antes) + pregunta + azar.choice(despues)
            pares.append((texto, con_estilo(respuestas[ident])))
    azar.shuffle(pares)
    return pares

In [ ]:
pares = ejemplos_de_entrenamiento(VARIANTES)
print(len(pares), "ejemplos. Dos de ellos:\n")
for pregunta, respuesta in pares[:2]:
    print("Cliente:", pregunta)
    print("Respuesta:", respuesta, "\n")

## 3. Solo cuenta la respuesta

In [ ]:
def codificar(tok, pregunta, respuesta):
    """Tokens de la conversación completa y etiquetas para la pérdida: -100 en
    la parte de la pregunta, para que solo se aprenda la respuesta."""
    mensajes = [{"role": "user", "content": pregunta}]
    previo = tok.apply_chat_template(mensajes, tokenize=False,
                                     add_generation_prompt=True)
    completo = previo + respuesta + "<|im_end|>"
    ids = tok(completo, add_special_tokens=False)["input_ids"]
    n_previo = len(tok(previo, add_special_tokens=False)["input_ids"])
    etiquetas = [-100] * n_previo + ids[n_previo:]
    return ids, etiquetas


def lotes(tok, pares, tamano=8):
    """Grupos de ejemplos rellenados hasta la misma longitud."""
    for i in range(0, len(pares), tamano):
        codificados = [codificar(tok, p, r) for p, r in pares[i:i + tamano]]
        largo = max(len(ids) for ids, _ in codificados)
        entrada, etiquetas, mascara = [], [], []
        for ids, et in codificados:
            relleno = largo - len(ids)
            entrada.append(ids + [tok.pad_token_id] * relleno)
            etiquetas.append(et + [-100] * relleno)
            mascara.append([1] * len(ids) + [0] * relleno)
        yield (torch.tensor(entrada, device=DISPOSITIVO),
               torch.tensor(etiquetas, device=DISPOSITIVO),
               torch.tensor(mascara, device=DISPOSITIVO))

In [ ]:
tok, modelo = cargar(MODELO_AJUSTE)
ids, etiquetas = codificar(tok, *pares[0])
print(len(ids), "tokens, de los que cuentan en la pérdida", sum(e != -100 for e in etiquetas))
print("Parte que se aprende:", repr(tok.decode([t for t, e in zip(ids, etiquetas) if e != -100])))

## 4. Probar antes de ajustar

In [ ]:
def responder(tok, modelo, pregunta, sistema=None, max_tokens=120):
    """Respuesta del modelo a una pregunta, con temperatura 0."""
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [
        {"role": "user", "content": pregunta}]
    texto = tok.apply_chat_template(mensajes, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(DISPOSITIVO)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    return tok.decode(salida[0, entrada["input_ids"].shape[1]:],
                      skip_special_tokens=True).strip()


INSTRUCCION_ESTILO = (
    "Eres el asistente de Ciclos Norte, una tienda de bicicletas. Responde en "
    "español, tuteando, en una o dos frases. Empieza siempre por «¡Hola!» y "
    "termina siempre con la firma «Un saludo,» y, en la línea siguiente, «El "
    "equipo de Ciclos Norte».")


# Diez preguntas sobre datos que aparecen en el entrenamiento, con otras
# palabras, y el dato que debe contener la respuesta; y diez sobre cosas que
# el entrenamiento no trata.
CONOCIDAS = [
    ("¿En cuántos días llega un envío normal a Madrid?", "48-72"),
    ("¿Qué me cobráis de portes si el pedido es de 20 euros?", "4,95"),
    ("¿Mandáis paquetes a Lanzarote?", "5-7"),
    ("¿Hasta cuándo puedo devolver algo que no me gusta?", "30 días"),
    ("¿Cuánto tardáis en devolverme el dinero?", "14 días"),
    ("¿Cuántos años de garantía tiene una eléctrica?", "2 años"),
    ("¿Se puede pagar con Bizum?", "Bizum"),
    ("¿Desde qué importe se puede financiar una compra?", "300"),
    ("¿De cuánto son las tarjetas regalo?", "25"),
    ("¿Cuánto tarda en cargarse del todo la batería?", "4 y 6"),
]


NUEVAS = [
    "¿Cuánto cuesta la Aero 3?", "¿Tenéis bicis de carretera de carbono?",
    "¿Dónde está mi pedido CN-10871?", "¿Organizáis rutas en grupo?",
    "¿Vendéis patinetes eléctricos?",
    "¿Qué presión deben llevar las ruedas de una bici de montaña?",
    "¿Hacéis descuentos a estudiantes?", "¿Puedo aparcar cerca de la tienda?",
    "¿Qué casco me recomendáis para ir al trabajo?",
    "¿Cuánto pesa la Plegable 20?"]


def comprobar_estilo(respuesta):
    """¿Empieza por el saludo, termina con la firma y es breve?"""
    return {"saludo": respuesta.startswith(SALUDO),
            "firma": respuesta.rstrip(".").endswith("El equipo de Ciclos Norte"),
            "breve": len(respuesta.split()) <= 70}


def evaluar(tok, modelo, sistema=None):
    """Estilo en las veinte preguntas y datos correctos en las diez conocidas."""
    filas = []
    for pregunta, dato in CONOCIDAS + [(p, None) for p in NUEVAS]:
        r = responder(tok, modelo, pregunta, sistema)
        filas.append({"pregunta": pregunta, "respuesta": r, **comprobar_estilo(r),
                      "dato": None if dato is None else dato in r})
    return filas


def resumen(filas):
    """Cuántas respuestas cumplen cada criterio."""
    return {"saludo": sum(f["saludo"] for f in filas),
            "firma": sum(f["firma"] for f in filas),
            "breve": sum(f["breve"] for f in filas),
            "datos": sum(bool(f["dato"]) for f in filas)}

In [ ]:
resultados = {}
resultados["0,5B sin instrucción"] = evaluar(tok, modelo)
resultados["0,5B con instrucción"] = evaluar(tok, modelo, INSTRUCCION_ESTILO)
for nombre, filas in resultados.items():
    print(f"{nombre:24s}", resumen(filas))

## 5. Ajustar con LoRA

In [ ]:
def ajustar(tok, modelo, pares, epocas=3, tasa=2e-4, rango=8):
    """Añade matrices LoRA de rango «rango» a las capas lineales de la atención
    y del MLP, y entrena solo esas matrices."""
    config = LoraConfig(r=rango, lora_alpha=2 * rango, lora_dropout=0.05,
                        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                        "gate_proj", "up_proj", "down_proj"],
                        task_type="CAUSAL_LM")
    modelo = get_peft_model(modelo, config)
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    total = sum(p.numel() for p in modelo.parameters())
    print(f"Parámetros entrenables: {entrenables:,} de {total:,} "
          f"({100 * entrenables / total:.2f} %)")
    optimizador = torch.optim.AdamW(
        [p for p in modelo.parameters() if p.requires_grad], lr=tasa)
    torch.manual_seed(SEMILLA)
    modelo.train()
    perdidas, inicio = [], time.time()
    for epoca in range(epocas):
        for entrada, etiquetas, mascara in lotes(tok, pares):
            perdida = modelo(input_ids=entrada, attention_mask=mascara,
                             labels=etiquetas).loss
            perdida.backward()
            optimizador.step()
            optimizador.zero_grad()
            perdidas.append(perdida.item())
        print(f"Época {epoca + 1}: pérdida media "
              f"{sum(perdidas[-len(pares) // 8:]) / (len(pares) // 8):.3f}")
    modelo.eval()
    return modelo, {"perdidas": perdidas, "segundos": time.time() - inicio,
                    "entrenables": entrenables, "total": total}

In [ ]:
modelo, entrenamiento = ajustar(tok, modelo, pares, EPOCAS)
print(f"Entrenamiento: {entrenamiento['segundos']:.0f} s")
resultados["0,5B ajustado con LoRA"] = evaluar(tok, modelo)
print(resumen(resultados["0,5B ajustado con LoRA"]))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(entrenamiento["perdidas"])
plt.xlabel("Paso de entrenamiento")
plt.ylabel("Pérdida")
plt.show()

Las respuestas del modelo ajustado. Fíjate en el estilo y en los datos, sobre todo en las preguntas nuevas:

In [ ]:
for f in resultados["0,5B ajustado con LoRA"]:
    print("•", f["pregunta"])
    print("  ", f["respuesta"].replace("\n", " / "))

## 6. Guardar y cargar el adaptador

In [ ]:
def cargar_ajustado(carpeta="adaptador_ciclos_norte"):
    """El modelo base con el adaptador LoRA guardado."""
    tok, base = cargar(MODELO_AJUSTE)
    return tok, PeftModel.from_pretrained(base, carpeta).eval()

In [ ]:
modelo.save_pretrained("adaptador_ciclos_norte")
import os
print({f: f"{os.path.getsize(os.path.join('adaptador_ciclos_norte', f)) / 1e6:.1f} MB"
       for f in os.listdir("adaptador_ciclos_norte")})

## 7. Comparar con el modelo grande (solo con GPU)

In [ ]:
if CON_GPU:
    del modelo
    torch.cuda.empty_cache()
    tok2, referencia = cargar(MODELO_REFERENCIA)
    resultados["1,5B con instrucción"] = evaluar(tok2, referencia, INSTRUCCION_ESTILO)
for nombre, filas in resultados.items():
    print(f"{nombre:24s}", resumen(filas))

## Ejercicios

1. Explica con tus palabras qué es un adaptador LoRA y por qué ocupa tan poco. ¿Qué ventaja tiene guardarlo aparte del modelo?

2. Calcula cuántos parámetros añade LoRA con r = 8 a una matriz de 896 × 4864, como las del MLP del modelo. ¿Qué fracción de la matriz es?

3. Ejecuta el ajuste sin GPU con `variantes=1` (60 ejemplos) y una época. ¿Cuánto tarda? ¿Aprende el estilo?

4. Repite el ajuste con 1 y con 2 épocas. ¿Aprende el estilo? ¿Cuántos datos acierta? ¿Hay un punto en que se aprende el estilo sin memorizar tanto?

5. Prueba con rangos de 2 y de 32. ¿Cambia el resultado? ¿Y el número de parámetros entrenables?

6. Escribe diez ejemplos de entrenamiento con un estilo diferente, por ejemplo más formal, de usted, y ajusta el modelo solo con ellos. ¿Bastan diez ejemplos para cambiar el estilo?

7. Añade a los datos de entrenamiento ejemplos de preguntas a las que la tienda no puede responder, con la respuesta «No tengo esa información; paso tu consulta al equipo», en el estilo de la tienda. ¿Inventa menos el modelo ajustado en las preguntas nuevas?

8. Usa el modelo ajustado con un mensaje de sistema que contenga las preguntas frecuentes completas. ¿Mantiene el estilo? ¿Acierta más datos? Compáralo con el modelo base con la misma información y la instrucción de estilo.

9. Carga el adaptador guardado con `cargar_ajustado` en una sesión nueva y comprueba que responde igual. Fusiona después el adaptador con el modelo con `merge_and_unload` y compara la velocidad de respuesta.

10. **Ficha técnica del ajuste.** Escribe una ficha breve del adaptador: datos de entrenamiento, configuración, tiempo, resultados de la evaluación y limitaciones conocidas, como la que debería acompañar a cualquier modelo que se pone en producción.

## Referencias

Modelos [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) y [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). LoRA: [Hu y colaboradores (2021)](https://arxiv.org/abs/2106.09685). QLoRA: [Dettmers y colaboradores (2023)](https://arxiv.org/abs/2305.14314). [PEFT](https://huggingface.co/docs/peft). Lección 11.08 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte y sus datos son ficticios. Código y experimentos propios.